# Studying the initial neutron star population

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import rcParams
from matplotlib import rc
import matplotlib as mpl
from scipy.integrate import quad
from pypopsyn.simulator.configuration import cfg
import pypopsyn.simulator.basics.constants as const

# Set `usetex=False' if you do not have LaTeX installed.
rc('text', usetex=False)
rc('font', family='serif')
mpl.rcParams['text.latex.preamble'] = [r"\usepackage{amsmath}"]

In [ ]:
rcParams["mathtext.fontset"] = "stix"
# rcParams["font.family"] = "Liberation serif"
rcParams["font.size"] = "22"
# rcParams['font.weight']='bold'
rcParams["figure.figsize"] = "8.0, 8.0"
rcParams["figure.autolayout"] = "False"

rcParams["axes.linewidth"] = "1.7"
rcParams["axes.labelpad"] = "15.0"
rcParams["axes.titlepad"] = "15.0"

rcParams["xtick.direction"] = "in"
rcParams["xtick.top"] = True
rcParams["xtick.major.pad"] = "10.0"
rcParams["xtick.minor.pad"] = "10.0"
rcParams["xtick.major.size"] = "10.0"
rcParams["xtick.major.width"] = "1.7"
rcParams["xtick.minor.size"] = "5.0"
rcParams["xtick.minor.width"] = "1.7"
rcParams["xtick.labelsize"] = "25"

rcParams["ytick.direction"] = "in"
rcParams["ytick.right"] = True
rcParams["ytick.major.pad"] = "10.0"
rcParams["ytick.minor.pad"] = "10.0"
rcParams["ytick.major.size"] = "10.0"
rcParams["ytick.major.width"] = "1.7"
rcParams["ytick.minor.size"] = "5.0"
rcParams["ytick.minor.width"] = "1.7"
rcParams["ytick.labelsize"] = "25"

KPC_TO_KM = 3.08567758e16  # Convert from [kpc] to [km].
YR_TO_S = 3600 * 24 * 365  # Convert from [yr] to [s].

Select an `initial_population.pkl.gz` file to import:

In [ ]:
data = pd.read_pickle("../data/initial_population.pkl.gz", compression="gzip")
data.head()

In [ ]:
age = data["age"]["[yr]"].to_numpy()
x = data["x"]["[kpc]"].to_numpy()
y = data["y"]["[kpc]"].to_numpy()
z = data["z"]["[kpc]"].to_numpy()
vk_r = data["vk_r"]["[kpc/yr]"].to_numpy() * KPC_TO_KM / YR_TO_S
vk_phi = data["vk_phi"]["[kpc/yr]"].to_numpy() * KPC_TO_KM / YR_TO_S
vk_z = data["vk_z"]["[kpc/yr]"].to_numpy() * KPC_TO_KM / YR_TO_S
v_orb = data["v_orb"]["[kpc/yr]"].to_numpy() * KPC_TO_KM / YR_TO_S
B = data["B"]["[G]"].to_numpy()
chi = data["chi"]["[rad]"].to_numpy()
P = data["P"]["[s]"].to_numpy()
P_dot = data["P_dot"]["[s/yr]"].to_numpy()

## Age information

Min and max ages:

In [ ]:
print(min(age), max(age))

Mean age of the pulsars

In [ ]:
t_age_mean = np.sum(age) / len(age)
print(t_age_mean)

In [ ]:
cfg["t_age_max"]

## Positional information

Top view of the galactic plane

In [ ]:
fig, ax = plt.subplots()

ax.plot(
    x,
    y,
    linestyle="None",
    marker="o",
    color="blue",
    markersize=1,
    alpha=0.2,
    rasterized=False
)

ax.plot(0.0, 8.5, marker="o", color="gold", markersize=6)
ax.set_xlabel(r"$x$ [kpc]")
ax.set_ylabel(r"$y$ [kpc]")
ax.set_xlim(-20.0, 20.0)
ax.set_ylim(-20.0, 20.0)

plt.show()

Side view of the galactic plane

In [ ]:
fig, ax = plt.subplots()

ax.plot(
    x,
    z,
    linestyle="None",
    marker="o",
    color="blue",
    markersize=1,
    alpha=0.3,
    rasterized=True,
)

ax.plot(0.0, 0.02, marker="o", color="gold", markersize=6)
ax.set_xlabel(r"$x$ [kpc]")
ax.set_ylabel(r"$z$ [kpc]")
ax.set_xlim(-20.0, 20.0)
ax.set_ylim(-20.0, 20.0)

plt.show()

Histrogramming the pulsars position and comparing to underlying PDF

In [ ]:
def pdf_r(r):
    "pdf from the Milky Way stellar surface density (Yusifov & Küçük 2004)"
    A = 37.6  # [stars kpc^-2]
    R1 = 0.55  # +-0.1 [kpc]
    Rsun = 8.5
    a = 1.64  # +-0.11
    b = 4.01  # +-0.24
    rho = (
        A
        * ((r + R1) / (Rsun + R1)) ** a
        * np.exp(-b * ((r - Rsun) / (Rsun + R1)))
    )
    pdf_r = 2 * np.pi * r * rho
    return pdf_r

For normalization purposes, determine the area underneath the theoretical PDF curve:

In [ ]:
pdf_area = quad(pdf_r, 0, 100)[0]
print(pdf_area)

In [ ]:
r = np.sqrt(x**2 + y**2)

In [ ]:
r_bins = np.linspace(0.0, 30.0, 51)

In [ ]:
fig, ax = plt.subplots()

ax.hist(
    r,
    bins=r_bins,
    histtype="step",
    edgecolor="blue",
    lw=4,
    alpha=0.5,
    label="initial simulation",
    density=True
)
ax.plot(
    r_bins,
    pdf_r(r_bins) / pdf_area,
    linestyle="-",
    lw=4,
    color="red",
    alpha=0.5,
    label="theoretical YK04",
)
plt.xlabel(r"$r$ [kpc]")
plt.ylabel(r"normalized radial PDF")
plt.xlim(0.0, 30.0)
plt.legend(frameon=False, loc=1)

plt.show()

## Proper velocity information

Histogrammed proper velocity components

In [ ]:
fig, ax = plt.subplots()
x_bins = np.linspace(-2000.,2000.,50)  

ax.hist(
    vk_r,
    bins=x_bins,
    histtype="step",
    edgecolor="red",
    lw=4,
    alpha=0.5,
    label=r"$v_{\rm{p,}r}$",
)
ax.hist(
    vk_phi,
    bins=x_bins,
    histtype="step",
    edgecolor="blue",
    lw=4,
    alpha=0.5,
    label=r"$v_{\rm{p,}\phi}$",
)
ax.hist(
    vk_z,
    bins=x_bins,
    histtype="step",
    edgecolor="green",
    lw=4,
    alpha=0.5,
    label=r"$v_{\rm{p,}z}$",
)
ax.set_xlabel(r"proper velocity components [km s$^{-1}$]")
ax.set_ylabel(r"number of NS")
plt.legend(frameon=False, loc=0)

plt.show()

Orbital (angular) velocity due to galactic potential

In [ ]:
fig, ax = plt.subplots()

ax.plot(
    r,
    abs(v_orb),
    linestyle="None",
    marker="o",
    color="blue",
    markersize=1,
    alpha=0.3,
    rasterized=True,
)

ax.set_xlabel(r"$r$ [kpc]")
ax.set_ylabel(r"$v_{\rm orb}$ [km s$^{-1}$]")

plt.show()

## Magneto-rotational information

Histogramming the periods, magnetic fields and misalignment angles

In [ ]:
def Gaussian(x, mean, sigma):
    y = (
        1
        / (sigma * np.sqrt(2 * np.pi))
        * np.exp(-((x - mean) ** 2) / (2 * sigma ** 2))
    )
    return y

In [ ]:
P_bins = np.linspace(-0.5, 2.0, 101)
print(max(P), min(P))

In [ ]:
pdf_P_initial_area = quad(Gaussian, 0, 100, args=(cfg["P_initial_mean"], cfg["P_initial_sigma"]))[0]
print(pdf_P_initial_area)

In [ ]:
fig, ax = plt.subplots()

ax.hist(
    P,
    bins=P_bins,
    histtype="step",
    edgecolor="blue",
    lw=4,
    alpha=0.5,
    label="simulation",
    density=True
)
ax.plot(P_bins, 
        Gaussian(
            P_bins, 
            cfg["P_initial_mean"], 
            cfg["P_initial_sigma"]
        ),
        linestyle="-",
        lw=4,
        color="red",
        alpha=0.5,
        label="theoretical",
)
ax.plot(P_bins[P_bins > 0], 
        Gaussian(
            P_bins[P_bins > 0], 
            cfg["P_initial_mean"], 
            cfg["P_initial_sigma"]
        ) / pdf_P_initial_area,
        linestyle="-",
        lw=4,
        color="green",
        alpha=0.5,
        label="rescaled",
)
plt.xlabel(r"$P$ [s]")
plt.ylabel(r"normalized PDF")
plt.xlim(-0.5, 2.0)
plt.legend(frameon=False, loc=1)

plt.show()

In [ ]:
B_log10_bins = np.linspace(9.0, 17.0, 101)
print(max(np.log10(B)), min(np.log10(B)))

In [ ]:
fig, ax = plt.subplots()

ax.hist(
    np.log10(B),
    bins=B_log10_bins,
    histtype="step",
    edgecolor="blue",
    lw=4,
    alpha=0.5,
    label="simulation",
    density=True
)
ax.plot(B_log10_bins, 
        Gaussian(
            B_log10_bins, 
            cfg["B_initial_log10_mean"], 
            cfg["B_initial_log10_sigma"]
        ),
        linestyle="-",
        lw=4,
        color="red",
        alpha=0.5,
        label="theoretical",
)
plt.xlabel(r"log$_{10} B$ [G]")
plt.ylabel(r"normalized PDF")
plt.xlim(9., 17.0)
plt.legend(frameon=False, loc=1)

plt.show()

In [ ]:
chi_bins = np.linspace(0, np.pi / 2, 101)
print(max(chi), min(chi))

In [ ]:
fig, ax = plt.subplots()

ax.hist(
    chi,
    bins=chi_bins,
    histtype="step",
    edgecolor="blue",
    lw=4,
    alpha=0.5,
    label="simulation",
    density=True
)
ax.plot(chi_bins, 
        np.sin(chi_bins),
        linestyle="-",
        lw=4,
        color="red",
        alpha=0.5,
        label="theoretical",
)
plt.xlabel(r"$\chi$ [rad]")
plt.ylabel(r"normalized PDF")
plt.xlim(0., np.pi / 2)
plt.legend(frameon=False, loc=2)

plt.show()

Plotting the PPdot diagram of the initial pulsar population:

In [ ]:
len(P[::40])

In [ ]:
fig, ax = plt.subplots()

ax.loglog(P[::40], P_dot[::40] / const.YR_TO_S, '.', color='gray', ms=6)

plt.xlabel(r"$P$ [s]")
plt.ylabel(r"Pdot [s/s]")

plt.show()

Note that this is not a snap-shot in time of the pulsar population, but all initial parameter sets projected into the PPdot plane.